# Datawell Consultancy
## Project: Fintech Servicing Operations Analytics
### Phase 1 — Data Quality, Cleaning & Master Dataset
---
**Prepared by:** Datawell Consultancy
**Version:** 1.0

---

## Problem Statement

A fintech company operates a multi-channel customer support function handling thousands of tickets monthly across inbound calls, live chat, email, chatbot, outbound calls, and social media. The support data lives across four disconnected systems — ticket records, agent profiles, customer demographics, and channel interaction logs. The business cannot answer basic operational questions such as:

- What is our SLA compliance rate across channels and priority levels?
- Which issue types take the longest to resolve?
- How are individual agents performing against key metrics?
- Which channel delivers the best customer satisfaction and resolution rate?
- Where are escalations and repeat contacts concentrated?

**Engagement objective:** Profile, clean, and unify all four data sources into a single analytics layer that answers these questions reliably.

---

## Dataset Overview

| File | Records | Description |
|------|---------|-------------|
| `tickets.csv` | 15,000 | Core support tickets — channel, issue, priority, SLA, satisfaction |
| `agents.csv` | 100 | Agent master data — team, shift, experience, specialisation |
| `customers.csv` | 5,000 | Customer profiles — segment, account type, region, tenure |
| `channel_interactions.csv` | 15,000 | Channel level interaction details — duration, messages, bot handling |

## Step 0: Environment Setup

In [1]:
import os
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Base directory setup
BASE_DIR         = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
RAW_DATA_DIR     = os.path.join(BASE_DIR, 'data', 'raw')
CLEANED_DATA_DIR = os.path.join(BASE_DIR, 'data', 'cleaned')
DASHBOARD_DIR    = os.path.join(BASE_DIR, 'dashboard')
DOCS_DIR         = os.path.join(BASE_DIR, 'docs')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Chart settings
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Environment ready')

Environment ready


## Step 1: Load Raw Data

In [2]:
tickets_raw      = pd.read_csv(os.path.join(RAW_DATA_DIR, 'tickets.csv'))
agents_raw       = pd.read_csv(os.path.join(RAW_DATA_DIR, 'agents.csv'))
customers_raw    = pd.read_csv(os.path.join(RAW_DATA_DIR, 'customers.csv'))
interactions_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'channel_interactions.csv'))

print('All files loaded successfully')
print(f'Tickets       : {tickets_raw.shape[0]:,} rows x {tickets_raw.shape[1]} columns')
print(f'Agents        : {agents_raw.shape[0]:,} rows x {agents_raw.shape[1]} columns')
print(f'Customers     : {customers_raw.shape[0]:,} rows x {customers_raw.shape[1]} columns')
print(f'Interactions  : {interactions_raw.shape[0]:,} rows x {interactions_raw.shape[1]} columns')

All files loaded successfully
Tickets       : 15,000 rows x 19 columns
Agents        : 100 rows x 8 columns
Customers     : 5,000 rows x 9 columns
Interactions  : 15,000 rows x 11 columns


## Step 2: Data Quality Audit

In [3]:
def data_quality_report(df, name):
    """Generate a structured data quality report for any dataframe."""
    print(f'\n{"="*60}')
    print(f'  DATA QUALITY REPORT — {name.upper()}')
    print(f'{"="*60}')

    total_rows = len(df)
    print(f'\nShape      : {total_rows:,} rows x {len(df.columns)} columns')
    print(f'Duplicates : {df.duplicated().sum():,} rows ({df.duplicated().sum()/total_rows*100:.1f}%)')

    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total_rows * 100).round(2)

    quality_df = pd.DataFrame({
        'Column'        : df.columns,
        'Data Type'     : df.dtypes.values,
        'Null Count'    : null_counts.values,
        'Null %'        : null_pct.values,
        'Unique Values' : [df[col].nunique() for col in df.columns]
    })

    print(f'\nColumn Profile:')
    print(quality_df.to_string(index=False))
    return quality_df

tickets_quality      = data_quality_report(tickets_raw,      'Tickets')
agents_quality       = data_quality_report(agents_raw,       'Agents')
customers_quality    = data_quality_report(customers_raw,    'Customers')
interactions_quality = data_quality_report(interactions_raw, 'Channel Interactions')


  DATA QUALITY REPORT — TICKETS

Shape      : 15,000 rows x 19 columns
Duplicates : 0 rows (0.0%)

Column Profile:
                Column Data Type  Null Count  Null %  Unique Values
             Ticket_ID    object           0    0.00          15000
           Customer_ID    object           0    0.00           4763
              Agent_ID    object           0    0.00            100
               Channel    object           0    0.00              6
            Issue_Type    object           0    0.00             10
              Priority    object           0    0.00              4
                Status    object           0    0.00              4
          Created_Date    object           0    0.00           1096
          Created_Hour     int64           0    0.00             24
 Resolution_Time_Hours   float64           0    0.00            529
      SLA_Target_Hours     int64           0    0.00              4
          SLA_Breached     int64           0    0.00              2


## Step 3: Business Rules Validation

In [4]:
print('BUSINESS RULES VALIDATION')
print('='*60)

print('\nTICKETS TABLE:')
print(f'  Resolution time <= 0                 : {(tickets_raw["Resolution_Time_Hours"] <= 0).sum()}')
print(f'  SLA breached tickets                 : {tickets_raw["SLA_Breached"].sum():,} ({tickets_raw["SLA_Breached"].mean()*100:.1f}%)')
print(f'  Escalated tickets                    : {tickets_raw["Escalated"].sum():,} ({tickets_raw["Escalated"].mean()*100:.1f}%)')
print(f'  First contact resolved               : {tickets_raw["First_Contact_Resolved"].sum():,} ({tickets_raw["First_Contact_Resolved"].mean()*100:.1f}%)')
print(f'  Repeat contacts                      : {tickets_raw["Repeat_Contact"].sum():,} ({tickets_raw["Repeat_Contact"].mean()*100:.1f}%)')
print(f'  Satisfaction scores present          : {tickets_raw["Satisfaction_Score"].notna().sum():,}')
print(f'  Satisfaction scores missing          : {tickets_raw["Satisfaction_Score"].isna().sum():,} (pending/escalated tickets)')
print(f'\n  Ticket status distribution:')
for status, count in tickets_raw['Status'].value_counts().items():
    print(f'    {status:<15}: {count:,} ({count/len(tickets_raw)*100:.1f}%)')
print(f'\n  Channel distribution:')
for ch, count in tickets_raw['Channel'].value_counts().items():
    print(f'    {ch:<20}: {count:,} ({count/len(tickets_raw)*100:.1f}%)')
print(f'\n  Priority distribution:')
for p, count in tickets_raw['Priority'].value_counts().items():
    print(f'    {p:<15}: {count:,} ({count/len(tickets_raw)*100:.1f}%)')

print('\nAGENTS TABLE:')
print(f'  Senior agents                        : {agents_raw["Is_Senior"].sum()}')
print(f'  Experience range                     : {agents_raw["Experience_Years"].min()} to {agents_raw["Experience_Years"].max()} years')

print('\nINTERACTIONS TABLE:')
print(f'  Bot handled interactions             : {interactions_raw["Bot_Handled"].sum():,} ({interactions_raw["Bot_Handled"].mean()*100:.1f}%)')
print(f'  Transferred to agent                 : {interactions_raw["Transferred_To_Agent"].sum():,} ({interactions_raw["Transferred_To_Agent"].mean()*100:.1f}%)')
print(f'  Resolution achieved                  : {interactions_raw["Resolution_Achieved"].sum():,} ({interactions_raw["Resolution_Achieved"].mean()*100:.1f}%)')

BUSINESS RULES VALIDATION

TICKETS TABLE:
  Resolution time <= 0                 : 0
  SLA breached tickets                 : 3,305 (22.0%)
  Escalated tickets                    : 3,067 (20.4%)
  First contact resolved               : 11,263 (75.1%)
  Repeat contacts                      : 3,732 (24.9%)
  Satisfaction scores present          : 10,633
  Satisfaction scores missing          : 4,367 (pending/escalated tickets)

  Ticket status distribution:
    Resolved       : 8,487 (56.6%)
    Pending        : 2,214 (14.8%)
    Escalated      : 2,153 (14.4%)
    Closed         : 2,146 (14.3%)

  Channel distribution:
    Live Chat           : 3,836 (25.6%)
    Inbound Call        : 3,785 (25.2%)
    Social Media        : 1,891 (12.6%)
    Outbound Call       : 1,865 (12.4%)
    Email               : 1,836 (12.2%)
    Chatbot             : 1,787 (11.9%)

  Priority distribution:
    Medium         : 6,444 (43.0%)
    Low            : 4,178 (27.9%)
    High           : 2,216 (14.8%)
    

## Step 4: Statistical Summary

In [5]:
print('STATISTICAL SUMMARY — TICKETS')
print(tickets_raw[['Resolution_Time_Hours', 'SLA_Target_Hours',
                    'Satisfaction_Score']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — INTERACTIONS')
print(interactions_raw[['Duration_Minutes', 'Sentiment_Score']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — CUSTOMERS')
print(customers_raw[['Tenure_Years', 'Monthly_Txn_Count']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — AGENTS')
print(agents_raw[['Experience_Years']].describe().round(2).to_string())

STATISTICAL SUMMARY — TICKETS
       Resolution_Time_Hours  SLA_Target_Hours  Satisfaction_Score
count              15,000.00         15,000.00           10,633.00
mean                   20.21             25.15                2.98
std                    25.76             16.33                1.15
min                     0.50              2.00                1.00
25%                     3.80              8.00                2.00
50%                    11.50             24.00                3.00
75%                    24.52             48.00                4.00
max                   144.00             48.00                5.00

STATISTICAL SUMMARY — INTERACTIONS
       Duration_Minutes  Sentiment_Score
count         15,000.00        15,000.00
mean              15.32            -0.00
std               11.71             0.58
min                0.50            -1.00
25%                5.30            -0.50
50%               12.00            -0.01
75%               23.70             0.50
max

## Step 5: Data Cleaning

In [6]:
# ─────────────────────────────────────────
# CLEAN TICKETS TABLE
# ─────────────────────────────────────────
tickets = tickets_raw.copy()

# Parse date
tickets['Created_Date'] = pd.to_datetime(tickets['Created_Date'])

# SLA deviation — how far over or under SLA
tickets['SLA_Deviation_Hours'] = (tickets['Resolution_Time_Hours'] - tickets['SLA_Target_Hours']).round(2)

# Resolution speed tier
tickets['Resolution_Speed_Tier'] = pd.cut(
    tickets['Resolution_Time_Hours'],
    bins=[0, 2, 8, 24, 48, float('inf')],
    labels=['Within 2hrs', 'Within 8hrs', 'Within 24hrs', 'Within 48hrs', 'Beyond 48hrs']
)

# Hour of day tier
tickets['Time_Of_Day'] = pd.cut(
    tickets['Created_Hour'],
    bins=[-1, 6, 12, 18, 23],
    labels=['Night (0-6)', 'Morning (6-12)', 'Afternoon (12-18)', 'Evening (18-24)']
)

# Satisfaction tier
tickets['Satisfaction_Tier'] = pd.cut(
    tickets['Satisfaction_Score'],
    bins=[0, 2.0, 3.0, 4.0, 5.0],
    labels=['Poor', 'Average', 'Good', 'Excellent']
)

# Is resolved flag
tickets['Is_Resolved'] = tickets['Status'].isin(['Resolved', 'Closed']).astype(int)

print(f'Tickets cleaned     : {len(tickets):,} records')
print(f'New columns added   : SLA_Deviation_Hours, Resolution_Speed_Tier, Time_Of_Day, Satisfaction_Tier, Is_Resolved')
tickets.head(3)

Tickets cleaned     : 15,000 records
New columns added   : SLA_Deviation_Hours, Resolution_Speed_Tier, Time_Of_Day, Satisfaction_Tier, Is_Resolved


,Ticket_ID,Customer_ID,Agent_ID,Channel,Issue_Type,Priority,Status,Created_Date,Created_Hour,Resolution_Time_Hours,SLA_Target_Hours,SLA_Breached,Escalated,First_Contact_Resolved,Satisfaction_Score,Repeat_Contact,Created_Year,Created_Month,Created_Quarter,SLA_Deviation_Hours,Resolution_Speed_Tier,Time_Of_Day,Satisfaction_Tier,Is_Resolved
0,TKT000001,CUST03981,AGT048,Social Media,Payment Failure,Medium,Resolved,2023-11-27,14,6.90,24,0,0,1,2.00,0,2023,11,4,-17.10,Within 8hrs,Afternoon (12-18),Poor,1
1,TKT000002,CUST02406,AGT049,Inbound Call,Statement Request,Medium,Closed,2024-12-16,22,17.10,24,0,0,1,2.70,1,2024,12,4,-6.90,Within 24hrs,Evening (18-24),Average,1
2,TKT000003,CUST01420,AGT098,Social Media,Account Locked,Medium,Escalated,2024-06-04,0,61.00,24,1,0,1,NaN,0,2024,6,2,37.00,Beyond 48hrs,Night (0-6),NaN,0


In [7]:
# ─────────────────────────────────────────
# CLEAN AGENTS TABLE
# ─────────────────────────────────────────
agents = agents_raw.copy()

agents['Hire_Date'] = pd.to_datetime(agents['Hire_Date'])

# Experience tier
agents['Experience_Tier'] = pd.cut(
    agents['Experience_Years'],
    bins=[0, 2, 5, 8, float('inf')],
    labels=['Junior', 'Mid', 'Senior', 'Expert']
)

print(f'Agents cleaned      : {len(agents):,} records')
print(f'New columns added   : Experience_Tier')
agents.head(3)

Agents cleaned      : 100 records
New columns added   : Experience_Tier


,Agent_ID,Agent_Name,Team,Shift,Experience_Years,Specialisation,Hire_Date,Is_Senior,Experience_Tier
0,AGT001,Ricky Salinas,Transactions,Morning,7,App Technical Issue,2023-05-07,1,Senior
1,AGT002,Heather Vasquez,Transactions,Night,4,Loan Query,2023-08-22,0,Mid
2,AGT003,Gregory Cox,Technical Support,Night,8,Refund Request,2023-12-16,0,Senior


In [8]:
# ─────────────────────────────────────────
# CLEAN CUSTOMERS TABLE
# ─────────────────────────────────────────
customers = customers_raw.copy()

# Tenure tier
customers['Tenure_Tier'] = pd.cut(
    customers['Tenure_Years'],
    bins=[0, 1, 3, 7, float('inf')],
    labels=['New (0-1yr)', 'Growing (1-3yr)', 'Established (3-7yr)', 'Loyal (7yr+)']
)

# Transaction activity tier
customers['Activity_Tier'] = pd.cut(
    customers['Monthly_Txn_Count'],
    bins=[0, 10, 25, 40, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print(f'Customers cleaned   : {len(customers):,} records')
print(f'New columns added   : Tenure_Tier, Activity_Tier')
customers.head(3)

Customers cleaned   : 5,000 records
New columns added   : Tenure_Tier, Activity_Tier


,Customer_ID,Customer_Name,Customer_Segment,Account_Type,Tenure_Years,Region,Is_Premium,Monthly_Txn_Count,Credit_Score_Band,Tenure_Tier,Activity_Tier
0,CUST00001,Vickie Fuller,Retail,Business,10.50,North,0,34,Fair,Loyal (7yr+),High
1,CUST00002,Sarah Pierce,Retail,Business,4.40,South,1,46,Poor,Established (3-7yr),Very High
2,CUST00003,Brett Vega,Corporate,Salary,4.00,South,0,3,Good,Established (3-7yr),Low


In [9]:
# ─────────────────────────────────────────
# CLEAN INTERACTIONS TABLE
# ─────────────────────────────────────────
interactions = interactions_raw.copy()

interactions['Interaction_Date'] = pd.to_datetime(interactions['Interaction_Date'])

# Duration tier
interactions['Duration_Tier'] = pd.cut(
    interactions['Duration_Minutes'],
    bins=[0, 5, 15, 30, float('inf')],
    labels=['Quick (0-5m)', 'Standard (5-15m)', 'Extended (15-30m)', 'Long (30m+)']
)

# Sentiment tier
interactions['Sentiment_Tier'] = pd.cut(
    interactions['Sentiment_Score'],
    bins=[-1.01, -0.3, 0.3, 1.01],
    labels=['Negative', 'Neutral', 'Positive']
)

print(f'Interactions cleaned: {len(interactions):,} records')
print(f'New columns added   : Duration_Tier, Sentiment_Tier')
interactions.head(3)

Interactions cleaned: 15,000 records
New columns added   : Duration_Tier, Sentiment_Tier


,Interaction_ID,Ticket_ID,Channel,Interaction_Date,Duration_Minutes,Messages_Exchanged,Bot_Handled,Transferred_To_Agent,Resolution_Achieved,Hold_Time_Minutes,Sentiment_Score,Duration_Tier,Sentiment_Tier
0,INT000001,TKT000001,Social Media,2023-11-27,9.40,5.00,0,0,1,NaN,-0.86,Standard (5-15m),Negative
1,INT000002,TKT000002,Inbound Call,2024-12-16,22.20,NaN,1,0,1,5.60,0.92,Extended (15-30m),Positive
2,INT000003,TKT000003,Social Media,2024-06-04,6.60,10.00,0,1,1,NaN,0.04,Standard (5-15m),Neutral


## Step 6: Build Unified Master Dataset

In [10]:
master = duckdb.query("""
    SELECT
        -- Ticket fields
        t.Ticket_ID,
        t.Created_Date,
        t.Created_Year,
        t.Created_Month,
        t.Created_Quarter,
        t.Created_Hour,
        t.Time_Of_Day,
        t.Channel,
        t.Issue_Type,
        t.Priority,
        t.Status,
        t.Is_Resolved,
        t.Resolution_Time_Hours,
        t.SLA_Target_Hours,
        t.SLA_Breached,
        t.SLA_Deviation_Hours,
        t.Resolution_Speed_Tier,
        t.Escalated,
        t.First_Contact_Resolved,
        t.Satisfaction_Score,
        t.Satisfaction_Tier,
        t.Repeat_Contact,

        -- Agent fields
        a.Agent_ID,
        a.Agent_Name,
        a.Team,
        a.Shift,
        a.Experience_Years,
        a.Experience_Tier,
        a.Specialisation,
        a.Is_Senior,

        -- Customer fields
        c.Customer_ID,
        c.Customer_Segment,
        c.Account_Type,
        c.Tenure_Years,
        c.Tenure_Tier,
        c.Region,
        c.Is_Premium,
        c.Monthly_Txn_Count,
        c.Activity_Tier,
        c.Credit_Score_Band,

        -- Interaction fields
        i.Interaction_ID,
        i.Duration_Minutes,
        i.Duration_Tier,
        i.Messages_Exchanged,
        i.Bot_Handled,
        i.Transferred_To_Agent,
        i.Resolution_Achieved,
        i.Hold_Time_Minutes,
        i.Sentiment_Score,
        i.Sentiment_Tier

    FROM tickets t
    LEFT JOIN agents       a ON t.Agent_ID    = a.Agent_ID
    LEFT JOIN customers    c ON t.Customer_ID = c.Customer_ID
    LEFT JOIN interactions i ON t.Ticket_ID   = i.Ticket_ID
""").df()

print(f'Master dataset built  : {len(master):,} rows x {len(master.columns)} columns')
print(f'Join coverage agents  : {master["Agent_ID"].notna().mean()*100:.1f}%')
print(f'Join coverage customers: {master["Customer_ID"].notna().mean()*100:.1f}%')
print(f'Join coverage interactions: {master["Interaction_ID"].notna().mean()*100:.1f}%')
master.head(3)

Master dataset built  : 15,000 rows x 50 columns
Join coverage agents  : 100.0%
Join coverage customers: 100.0%
Join coverage interactions: 100.0%


,Ticket_ID,Created_Date,Created_Year,Created_Month,Created_Quarter,Created_Hour,Time_Of_Day,Channel,Issue_Type,Priority,Status,Is_Resolved,Resolution_Time_Hours,SLA_Target_Hours,SLA_Breached,SLA_Deviation_Hours,Resolution_Speed_Tier,Escalated,First_Contact_Resolved,Satisfaction_Score,Satisfaction_Tier,Repeat_Contact,Agent_ID,Agent_Name,Team,Shift,Experience_Years,Experience_Tier,Specialisation,Is_Senior,Customer_ID,Customer_Segment,Account_Type,Tenure_Years,Tenure_Tier,Region,Is_Premium,Monthly_Txn_Count,Activity_Tier,Credit_Score_Band,Interaction_ID,Duration_Minutes,Duration_Tier,Messages_Exchanged,Bot_Handled,Transferred_To_Agent,Resolution_Achieved,Hold_Time_Minutes,Sentiment_Score,Sentiment_Tier
0,TKT000001,2023-11-27,2023,11,4,14,Afternoon (12-18),Social Media,Payment Failure,Medium,Resolved,1,6.90,24,0,-17.10,Within 8hrs,0,1,2.00,Poor,0,AGT048,Briana Hunt,KYC & Compliance,Night,4,Mid,Transaction Dispute,0,CUST03981,Premium,NRI,5.80,Established (3-7yr),Central,1,36,High,Excellent,INT000001,9.40,Standard (5-15m),5.00,0,0,1,NaN,-0.86,Negative
1,TKT000002,2024-12-16,2024,12,4,22,Evening (18-24),Inbound Call,Statement Request,Medium,Closed,1,17.10,24,0,-6.90,Within 24hrs,0,1,2.70,Average,1,AGT049,Keith Terry,Transactions,Night,7,Senior,App Technical Issue,0,CUST02406,SME,Current,1.50,Growing (1-3yr),East,0,4,Low,Poor,INT000002,22.20,Extended (15-30m),NaN,1,0,1,5.60,0.92,Positive
2,TKT000003,2024-06-04,2024,6,2,0,Night (0-6),Social Media,Account Locked,Medium,Escalated,0,61.00,24,1,37.00,Beyond 48hrs,0,1,NaN,NaN,0,AGT098,Jerry Luna,Cards & Payments,Morning,7,Senior,Transaction Dispute,0,CUST01420,Corporate,Current,0.50,New (0-1yr),Central,0,43,Very High,Excellent,INT000003,6.60,Standard (5-15m),10.00,0,1,1,NaN,0.04,Neutral


## Step 7: Export Cleaned Data

In [11]:
tickets.to_csv(os.path.join(CLEANED_DATA_DIR, 'tickets_cleaned.csv'), index=False)
agents.to_csv(os.path.join(CLEANED_DATA_DIR, 'agents_cleaned.csv'), index=False)
customers.to_csv(os.path.join(CLEANED_DATA_DIR, 'customers_cleaned.csv'), index=False)
interactions.to_csv(os.path.join(CLEANED_DATA_DIR, 'interactions_cleaned.csv'), index=False)
master.to_csv(os.path.join(CLEANED_DATA_DIR, 'master_dataset.csv'), index=False)

print('All cleaned files exported to data/cleaned/')
print(f'  tickets_cleaned.csv       : {len(tickets):,} rows')
print(f'  agents_cleaned.csv        : {len(agents):,} rows')
print(f'  customers_cleaned.csv     : {len(customers):,} rows')
print(f'  interactions_cleaned.csv  : {len(interactions):,} rows')
print(f'  master_dataset.csv        : {len(master):,} rows')

All cleaned files exported to data/cleaned/
  tickets_cleaned.csv       : 15,000 rows
  agents_cleaned.csv        : 100 rows
  customers_cleaned.csv     : 5,000 rows
  interactions_cleaned.csv  : 15,000 rows
  master_dataset.csv        : 15,000 rows
